In [9]:
!pip install -q streamlit pyngrok groq

In [6]:
%%writefile app.py

import streamlit as st
from groq import Groq
import os

# Ambil API key dari environment
GROQ_API_KEY = os.environ.get("GROQ_API_KEY")

# Inisialisasi Groq client
client = Groq(api_key=GROQ_API_KEY)

st.set_page_config(
    page_title="EduPal - Chatbot Edukasi",
    page_icon="🎓",
    layout="centered",
)

st.markdown("""
<style>
    .main { background-color: #f0f4ff; }
    h1 { color: #4A90D9; text-align: center; }
    .subtitle { text-align: center; color: #888; font-size: 14px; margin-top: -15px; }
</style>
""", unsafe_allow_html=True)

st.title("🎓 EduPal")
st.markdown('<p class="subtitle">Your Friendly Learning Buddy — Powered by Groq AI</p>', unsafe_allow_html=True)
st.divider()

# System instruction untuk EduPal
SYSTEM_PROMPT = """Kamu adalah EduPal, chatbot edukasi yang ramah dan menyenangkan untuk semua kalangan.
Tugasmu adalah membantu pengguna belajar berbagai topik dengan cara yang santai, mudah dipahami, dan interaktif.

Panduan gaya bicara:
- Gunakan bahasa Indonesia yang santai dan friendly
- Sapa pengguna dengan hangat
- Berikan penjelasan yang sederhana tapi tetap akurat
- Gunakan emoji secukupnya agar terasa lebih hidup
- Jika ada soal/pertanyaan, bantu jelaskan langkah-langkahnya
- Selalu semangati pengguna untuk terus belajar
"""

# Inisialisasi session state
if "messages" not in st.session_state:
    st.session_state.messages = [
        {
            "role": "assistant",
            "content": "Halo! 👋 Aku **EduPal**, teman belajar kamu!\nMau belajar apa hari ini? Tanya aja, aku siap bantu! 😊",
        }
    ]

# Tampilkan pesan-pesan sebelumnya
for msg in st.session_state.messages:
    with st.chat_message(msg["role"], avatar="🎓" if msg["role"] == "assistant" else "🧑"):
        st.markdown(msg["content"])

# Input dari user
if prompt := st.chat_input("Tanya apa saja... 💬"):
    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user", avatar="🧑"):
        st.markdown(prompt)

    with st.chat_message("assistant", avatar="🎓"):
        with st.spinner("EduPal lagi mikir... 🤔"):
            try:
                # Siapkan history chat untuk Groq
                chat_history = []
                for msg in st.session_state.messages[:-1]:
                    chat_history.append({"role": msg["role"], "content": msg["content"]})

                # Kirim request ke Groq
                response = client.chat.completions.create(
                    model="llama-3.3-70b-versatile",  # Model gratis & cepat
                    messages=[
                        {"role": "system", "content": SYSTEM_PROMPT},
                        *chat_history,
                        {"role": "user", "content": prompt}
                    ],
                    temperature=0.7,
                    max_tokens=1024,
                )
                reply = response.choices[0].message.content
            except Exception as e:
                reply = f"Maaf, terjadi kesalahan: {str(e)}"

        st.markdown(reply)

    st.session_state.messages.append({"role": "assistant", "content": reply})

# Sidebar
with st.sidebar:
    st.header("⚙️ EduPal Info")
    st.markdown("""
    **🤖 Model:** Llama 3.3 70B (via Groq)
    **🎯 Use Case:** Edukasi & Belajar
    **👥 Target:** Semua Kalangan
    **🗣️ Bahasa:** Indonesia (Santai)
    """)
    st.divider()
    st.markdown("**💡 Contoh pertanyaan:**")
    st.markdown("""
    - Apa itu fotosintesis?
    - Jelaskan rumus luas lingkaran
    - Ceritakan sejarah kemerdekaan RI
    - Cara belajar yang efektif itu gimana?
    """)
    st.divider()
    if st.button("🗑️ Reset Chat"):
        st.session_state.messages = [
            {
                "role": "assistant",
                "content": "Halo! 👋 Aku **EduPal**, teman belajar kamu!\nMau belajar apa hari ini? 😊",
            }
        ]
        st.rerun()

Overwriting app.py


In [10]:
# ─── CELL 3: Setup API Key Groq ─────────────────
import os
from google.colab import userdata

# Ambil API Key dari Colab Secrets
GROQ_API_KEY = userdata.get("GROQ_API_KEY")

# Set environment variable
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

print("✅ Groq API Key berhasil dimuat!")
print(f"🔑 API Key (awal): {GROQ_API_KEY[:12]}...")
print(f"📏 Panjang key: {len(GROQ_API_KEY)} karakter")
print("💡 Key siap digunakan untuk EduPal!")

✅ Groq API Key berhasil dimuat!
🔑 API Key (awal): gsk_Ikl9Lour...
📏 Panjang key: 56 karakter
💡 Key siap digunakan untuk EduPal!


In [11]:
# ─── CELL 4 FINAL (Lengkapi dengan secret Anda) ─────────────────
from pyngrok import ngrok
import subprocess
import time
import os
from google.colab import userdata

# 1. Setup API Key Groq
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

# 2. Setup Authtoken ngrok
ngrok.set_auth_token(userdata.get("NGROK_AUTH_TOKEN"))
print("✅ Authtoken ngrok berhasil disetel")

# 3. Kill tunnel lama
print("Membersihkan tunnel ngrok lama...")
ngrok.kill()
time.sleep(2)

# 4. Jalankan Streamlit
print("Menjalankan Streamlit server...")
proc = subprocess.Popen(["streamlit", "run", "app.py", "--server.port=8501", "--server.headless=true"])
time.sleep(5)

# 5. Buat tunnel publik
print("Membuat tunnel ngrok...")
public_url = ngrok.connect(8501)

# 6. Tampilkan hasil
print("\n" + "=" * 60)
print("🎓 EduPal BERHASIL dijalankan!")
print("=" * 60)
print(f"📱 Buka link ini di browser: {public_url}")
print("=" * 60)
print("\n✅ Selamat belajar dengan EduPal! 🎉")

✅ Authtoken ngrok berhasil disetel
Membersihkan tunnel ngrok lama...
Menjalankan Streamlit server...
Membuat tunnel ngrok...

🎓 EduPal BERHASIL dijalankan!
📱 Buka link ini di browser: NgrokTunnel: "https://drizzly-tactile-clang.ngrok-free.dev" -> "http://localhost:8501"

✅ Selamat belajar dengan EduPal! 🎉
